In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import time
import duckdb

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✓ Bibliotecas importadas com sucesso")
print(f"  pandas  {pd.__version__}")
print(f"  duckdb  {duckdb.__version__}")

import pyarrow
print(f"  pyarrow {pyarrow.__version__}")

✓ Bibliotecas importadas com sucesso
  pandas  2.2.2
  duckdb  1.5.3
  pyarrow 24.0.0


In [ ]:
CURRENT_DIR = Path.cwd()
if CURRENT_DIR.name == "notebooks" and CURRENT_DIR.parent.name == "src":
    PROJECT_DIR = CURRENT_DIR.parent.parent
elif CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR
SAMPLE_DATA_DIR = PROJECT_DIR / "src" / "sample_data"
OUTPUT_DIR = CURRENT_DIR

print("Carregando arquivos Olist...")

t = time.time()
df_orders    = pd.read_csv(SAMPLE_DATA_DIR / "olist_orders_dataset.csv")
df_items     = pd.read_csv(SAMPLE_DATA_DIR / "olist_order_items_dataset.csv")
df_customers = pd.read_csv(SAMPLE_DATA_DIR / "olist_customers_dataset.csv")[["customer_id", "customer_state"]]
df_products  = pd.read_csv(SAMPLE_DATA_DIR / "olist_products_dataset.csv")[["product_id", "product_category_name"]]

df = (df_orders
      .merge(df_items,     on="order_id",    how="inner")
      .merge(df_customers, on="customer_id", how="left")
      .merge(df_products,  on="product_id",  how="left"))

print(f"✓ Carregado em {time.time()-t:.2f}s")
print(f"  Shape: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"  Colunas: {list(df.columns)}")
print()
print(df.head(3))

Carregando arquivos Olist...
✓ Carregado em 0.40s
  Shape: 112,650 linhas × 16 colunas
  Colunas: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'customer_state', 'product_category_name']

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   

  order_delivered_carrie

In [7]:
print("=== RESUMO DO DATASET ===")
print()

print(f"Linhas:   {df.shape[0]:>10,}")
print(f"Colunas:  {df.shape[1]:>10}")
print()

print("Tipos de dado:")
for col, dtype in df.dtypes.items():
    n_unique = df[col].nunique()
    n_null   = df[col].isna().sum()
    print(f"  {col:<40} {str(dtype):<12} {n_unique:>8,} únicos   {n_null:>6,} nulls")

print()

print("Primeiras 3 linhas:")
print(df[['order_id', 'product_category_name', 'price', 'customer_state']].head(3))

=== RESUMO DO DATASET ===

Linhas:      112,650
Colunas:          16

Tipos de dado:
  order_id                                 object         98,666 únicos        0 nulls
  customer_id                              object         98,666 únicos        0 nulls
  order_status                             object              7 únicos        0 nulls
  order_purchase_timestamp                 object         98,112 únicos        0 nulls
  order_approved_at                        object         90,174 únicos       15 nulls
  order_delivered_carrier_date             object         81,017 únicos    1,194 nulls
  order_delivered_customer_date            object         95,664 únicos    2,454 nulls
  order_estimated_delivery_date            object            450 únicos        0 nulls
  order_item_id                            int64              21 únicos        0 nulls
  product_id                               object         32,951 únicos        0 nulls
  seller_id                                ob

In [12]:
print("Carregando arquivos Olist...")

t = time.time()
df_orders    = pd.read_csv(SAMPLE_DATA_DIR / "olist_orders_dataset.csv")
df_items     = pd.read_csv(SAMPLE_DATA_DIR / "olist_order_items_dataset.csv")
df_customers = pd.read_csv(SAMPLE_DATA_DIR / "olist_customers_dataset.csv")[["customer_id", "customer_state"]]
df_products  = pd.read_csv(SAMPLE_DATA_DIR / "olist_products_dataset.csv")[["product_id", "product_category_name"]]

df = (df_orders
      .merge(df_items,     on="order_id",    how="inner")
      .merge(df_customers, on="customer_id", how="left")
      .merge(df_products,  on="product_id",  how="left"))

print(f"✓ Carregado em {time.time()-t:.2f}s")
print(f"  Shape: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"  Colunas: {list(df.columns)}")
print()
print(df.head(3))

print("\n=== PREDICATE PUSHDOWN — como o engine evita ler dados desnecessários ===\n")

arquivo_snappy = OUTPUT_DIR / "pedidos_snappy.parquet"
arquivo_ordenado = OUTPUT_DIR / "pedidos_ordenado.parquet"

df.to_parquet(arquivo_snappy, compression="snappy", index=False)

df_ordenado = df.sort_values("product_category_name")
df_ordenado.to_parquet(arquivo_ordenado, compression="snappy", index=False)

query_filtrada = f"""
    SELECT COUNT(*), SUM(price)
    FROM '{arquivo_snappy.as_posix()}'
    WHERE product_category_name = 'eletronicos'
"""

query_filtrada_ordenada = f"""
    SELECT COUNT(*), SUM(price)
    FROM '{arquivo_ordenado.as_posix()}'
    WHERE product_category_name = 'eletronicos'
"""

print("Arquivo NÃO ordenado:")
t = time.time()
r1 = duckdb.sql(query_filtrada).df()
t1 = time.time() - t
print(f"  Resultado: {r1.iloc[0,0]:,} pedidos, receita R$ {r1.iloc[0,1]:,.2f}")
print(f"  Tempo: {t1:.3f}s\n")

print("Arquivo ordenado por product_category_name:")
t = time.time()
r2 = duckdb.sql(query_filtrada_ordenada).df()
t2 = time.time() - t
print(f"  Resultado: {r2.iloc[0,0]:,} pedidos, receita R$ {r2.iloc[0,1]:,.2f}")
print(f"  Tempo: {t2:.3f}s")

print(f"\n  Aceleração com dados ordenados: {t1/t2:.1f}x")
print("  (com dataset pequeno a diferença é modesta — em TBs seria dramática)")

print()
print("=== EXPLAIN — plano de execução do DuckDB ===")
explain = duckdb.sql(f"EXPLAIN {query_filtrada_ordenada}").df()
print(explain.to_string(index=False))

Carregando arquivos Olist...
✓ Carregado em 0.40s
  Shape: 112,650 linhas × 16 colunas
  Colunas: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'customer_state', 'product_category_name']

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   

  order_delivered_carrie